# Dashboard LCA — adaptive plotting version

This version keeps the notebook as the single plotting/dashboard entry point, but replaces the fixed 48-slice plotting with adaptive figures.

What changed:

- It uses the active settings in `dashboard_config.py`, including `GRID_TIME_MODE = "single"`, `"range"`, or `"year_average"`.
- It loads both wind/grid output folders when `WIND_LCA_MODE = "both"`.
- It automatically switches between raw half-hourly plots, resampled daily/weekly/monthly plots, and categorical representative-day plots.
- It adds duration-curve and heatmap-style diagnostics so a full year does not become an unreadable spaghetti plot.
- It avoids hard-coded x-axis tick labels, so the same notebook works for one day, one week, one year, or the 12 representative `year_average` outputs.

Edit settings in `dashboard_config.py`, run the child notebooks if you need fresh outputs, then run the plotting cells below.


In [2]:
# All settings come from dashboard_config.py — the single master dashboard.
# Edit values there once; every notebook picks them up.
from pathlib import Path
from typing import Iterable, Optional, Sequence
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import PercentFormatter

from IPython.display import display

from dashboard_config import *
import dashboard_config as cfg

try:
    import lca_helpers as H
except Exception as exc:
    H = None
    warnings.warn(f"Could not import lca_helpers. Plotting still works, but child notebook execution may fail: {exc}")

print_dashboard()


Master Dashboard
----------------
Project:                 hydrogen-smr
Foreground DB:           hydrogen foreground
Build foreground DB:     True
Run reference LCA:       False
Run grid scenario LCA:   True
Run wind/grid LCA:       True
Grid method:             cheap | loss factor: 1.0316426921769999
Wind/grid method:        cheap
Selected grid techs:     ['AE operation']
Wind/grid mode:          both (blended + switching) | electrolyser(s): ['AE operation']


## Optional: re-run the building-block notebooks

By default this dashboard only loads the latest CSV outputs and plots them. Set the flags below to `True` only when you want to regenerate results.


In [ ]:
# Set to True only if you want to re-execute the child notebooks.
# For plotting existing CSVs, leave these as False.
RUN_FOREGROUND_NOTEBOOK = False
RUN_GRID_NOTEBOOK       = False
RUN_WIND_NOTEBOOK       = False

if any([RUN_FOREGROUND_NOTEBOOK, RUN_GRID_NOTEBOOK, RUN_WIND_NOTEBOOK]) and H is None:
    raise RuntimeError("lca_helpers could not be imported, so the child notebooks cannot be run from here.")

if RUN_FOREGROUND_NOTEBOOK:
    get_ipython().run_line_magic("run", '"3.tech_lca_foreground.ipynb"')
if RUN_GRID_NOTEBOOK:
    get_ipython().run_line_magic("run", '"4.custom_grid.ipynb"')
if RUN_WIND_NOTEBOOK:
    get_ipython().run_line_magic("run", '"5.wind_power.ipynb"')


Master Dashboard
----------------
Project:                 hydrogen-smr
Foreground DB:           hydrogen foreground
Build foreground DB:     True
Run reference LCA:       False
Run grid scenario LCA:   True
Run wind/grid LCA:       True
Grid method:             cheap | loss factor: 1.0316426921769999
Wind/grid method:        cheap
Selected grid techs:     ['AE operation']
Wind/grid mode:          both (blended + switching) | electrolyser(s): ['AE operation']

Current Brightway project: hydrogen-smr
Using ecoinvent database: ecoinvent-3.9.1-apos
Using biosphere database: ecoinvent-3.9.1-biosphere
Using foreground database: hydrogen foreground
Using LCIA method: ('ecoinvent-3.9.1', 'IPCC 2021 no LT', 'climate change no LT', 'global warming potential (GWP100) no LT')
Custom GB electricity mix — candidate indexes
---------------------------------------------
  GAS                    candidate #3
  COAL                   candidate #0
  NUCLEAR                candidate #1
  WIND_GT3_ONSHORE

ValueError: Image size of 1500x97600 pixels is too large. It must be less than 2^16 in each direction.

Error in callback <function _draw_all_if_interactive at 0x000001212AABE7A0> (for post_execute), with arguments args (),kwargs {}:


ValueError: Image size of 1500x97600 pixels is too large. It must be less than 2^16 in each direction.

ValueError: Image size of 1500x97600 pixels is too large. It must be less than 2^16 in each direction.

<Figure size 1500x97600 with 732 Axes>

ValueError: Image size of 1500x97600 pixels is too large. It must be less than 2^16 in each direction.

## Load latest result CSVs

This cell is deliberately flexible. It looks in the output folders defined by `dashboard_config.py`, parses whatever datetime-like column exists, and keeps both wind/grid modes when `WIND_LCA_MODE = "both"`.


In [ ]:
def latest_csv(folder: str | Path) -> Optional[Path]:
    """Return the most recently modified CSV in a folder, or None."""
    folder = Path(folder)
    if not folder.exists():
        return None
    files = sorted(folder.glob("*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0] if files else None


def read_csv_with_datetime(path: Path) -> pd.DataFrame:
    """Read a CSV and normalise its main datetime column to `datetime` when possible."""
    df = pd.read_csv(path)
    if df.empty:
        return df

    preferred = [
        "datetime", "time", "timestamp", "date",
        "selected_datetime", "representative_datetime", "representative_date",
        "grid_datetime", "ninja_datetime",
    ]
    candidates = [c for c in preferred if c in df.columns]
    candidates += [c for c in df.columns if c not in candidates and any(k in c.lower() for k in ["date", "time"])]

    best_col = None
    best_valid = 0
    for col in candidates:
        parsed = pd.to_datetime(df[col], errors="coerce", utc=False)
        valid = int(parsed.notna().sum())
        if valid > best_valid:
            best_col = col
            best_valid = valid

    if best_col is not None and best_valid > 0:
        df["datetime"] = pd.to_datetime(df[best_col], errors="coerce", utc=False)
    return df


def load_grid_results() -> tuple[Optional[pd.DataFrame], Optional[Path]]:
    path = latest_csv(cfg.GRID_OUTPUT_DIR)
    if path is None:
        return None, None
    df = read_csv_with_datetime(path)
    if "datetime" in df.columns:
        df = df.sort_values("datetime").set_index("datetime", drop=False)
    return df, path


def wind_modes_to_load() -> list[str]:
    mode = getattr(cfg, "WIND_LCA_MODE", "both")
    if mode == "both":
        return ["blended", "switching"]
    if mode in ("blended", "switching"):
        return [mode]
    return ["blended", "switching"]


def load_wind_results() -> tuple[Optional[pd.DataFrame], dict[str, Optional[Path]]]:
    folders = {
        "blended": Path(cfg.WIND_OUTPUT_DIR_BLENDED),
        "switching": Path(cfg.WIND_OUTPUT_DIR_SWITCHING),
    }
    paths: dict[str, Optional[Path]] = {}
    parts: list[pd.DataFrame] = []

    for mode in wind_modes_to_load():
        path = latest_csv(folders[mode])
        paths[mode] = path
        if path is None:
            continue
        df = read_csv_with_datetime(path)
        if df.empty:
            continue
        if "wind_lca_mode" not in df.columns:
            df["wind_lca_mode"] = mode
        parts.append(df)

    if not parts:
        return None, paths

    out = pd.concat(parts, ignore_index=True, sort=False)
    if "datetime" in out.columns:
        out = out.sort_values(["wind_lca_mode", "datetime"]).reset_index(drop=True)
    return out, paths


grid_df_results, grid_csv = load_grid_results()
wind_df_results, wind_csvs = load_wind_results()

print("Custom-grid CSV:   ", grid_csv if grid_csv is not None else "(none found)")
for mode, path in wind_csvs.items():
    print(f"Wind/grid CSV [{mode:9s}]:", path if path is not None else "(none found)")

if grid_df_results is not None:
    print(f"\nLoaded custom-grid rows: {len(grid_df_results):,}")
    display(grid_df_results.head())
else:
    print("\nNo custom-grid output found — run 4.custom_grid.ipynb first.")

if wind_df_results is not None:
    print(f"Loaded wind/grid rows:   {len(wind_df_results):,}")
    display(wind_df_results.head())
else:
    print("No wind/grid output found — run 5.wind_power.ipynb first.")


## Adaptive plotting helpers

These functions are used by the plots below. They do three important things:

1. Use categorical labels for `year_average` representative days.
2. Use normal date axes for short ranges.
3. Resample long half-hourly ranges before plotting, so full-year results remain readable.


In [ ]:
GWP_THRESHOLDS = [2.0]  # kg CO2eq/kg H2; add more if useful, e.g. [2.0, 4.0]

GRID_METADATA_COLS = {
    "datetime", "time", "timestamp", "date",
    "carbon_intensity", "custom_electricity_score", "custom_electricity_input_kwh_per_kwh",
    "season", "season_name", "representative", "representative_type", "rep_type", "rep_label",
    "representative_label", "day_type", "wind_day_type", "selected_date", "selected_datetime",
    "wind_total", "wind_share", "wind_fraction", "wind_rank", "year", "month", "day",
}


def has_datetime(df: pd.DataFrame) -> bool:
    return "datetime" in df.columns and pd.to_datetime(df["datetime"], errors="coerce").notna().any()


def as_datetime_series(df: pd.DataFrame) -> pd.Series:
    return pd.to_datetime(df["datetime"], errors="coerce")


def is_year_average_like(df: pd.DataFrame, *, source: str = "grid") -> bool:
    """Detect representative-day output even if only the CSV is available."""
    if source == "grid" and getattr(cfg, "GRID_TIME_MODE", None) == "year_average":
        return True
    if source == "wind" and getattr(cfg, "GRID_TIME_MODE", None) == "year_average":
        # The wind notebook reuses the grid time mode in the current workflow.
        return True
    lower_cols = {c.lower() for c in df.columns}
    if any("representative" in c for c in lower_cols):
        return True
    if {"season", "rep_type"}.issubset(lower_cols) or {"season", "day_type"}.issubset(lower_cols):
        return True
    return False


def first_existing_col(df: pd.DataFrame, names: Sequence[str]) -> Optional[str]:
    lookup = {c.lower(): c for c in df.columns}
    for name in names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    return None


def representative_labels(df: pd.DataFrame) -> list[str]:
    """Create readable labels for single/range/year_average data."""
    data = df.reset_index(drop=True).copy()

    label_col = first_existing_col(data, [
        "rep_label", "representative_label", "representative_day_label", "label"
    ])
    if label_col is not None:
        labels = data[label_col].astype(str).tolist()
    else:
        season_col = first_existing_col(data, ["season", "season_name"])
        type_col = first_existing_col(data, [
            "rep_type", "representative_type", "day_type", "wind_day_type", "representative"
        ])

        date_part = None
        if has_datetime(data):
            dt = as_datetime_series(data)
            # Include the year only when it helps distinguish points.
            fmt = "%d %b %Y" if dt.dt.year.nunique(dropna=True) > 1 else "%d %b"
            date_part = dt.dt.strftime(fmt)

        if season_col is not None and type_col is not None:
            labels = (data[season_col].astype(str) + "\n" + data[type_col].astype(str)).tolist()
        elif type_col is not None and date_part is not None:
            labels = (data[type_col].astype(str) + "\n" + date_part.astype(str)).tolist()
        elif season_col is not None and date_part is not None:
            labels = (data[season_col].astype(str) + "\n" + date_part.astype(str)).tolist()
        elif date_part is not None:
            labels = date_part.astype(str).tolist()
        else:
            labels = [f"slice {i + 1}" for i in range(len(data))]

    # Make duplicate labels unambiguous without making normal labels ugly.
    seen: dict[str, int] = {}
    out = []
    for lab in labels:
        count = seen.get(lab, 0) + 1
        seen[lab] = count
        out.append(lab if count == 1 else f"{lab}\n({count})")
    return out


def grid_tech_columns(df: pd.DataFrame) -> list[str]:
    """Find LCA result columns in the custom-grid output."""
    configured = [t for t in getattr(cfg, "SELECTED_LCA_TECHS", []) if t in df.columns]
    if configured:
        return configured

    numeric_cols = list(df.select_dtypes(include="number").columns)
    tech_cols = []
    for col in numeric_cols:
        low = col.lower()
        if col in GRID_METADATA_COLS or low in GRID_METADATA_COLS:
            continue
        if any(token in low for token in [
            "score", "carbon", "intensity", "input", "kwh", "wind", "grid",
            "fraction", "share", "capacity", "power", "rank", "year", "month", "day"
        ]):
            continue
        tech_cols.append(col)
    return tech_cols


def choose_secondary_grid_col(df: pd.DataFrame) -> tuple[Optional[str], Optional[str]]:
    if "custom_electricity_score" in df.columns and pd.to_numeric(df["custom_electricity_score"], errors="coerce").notna().any():
        return "custom_electricity_score", "Electricity score (kg CO2eq/kWh)"
    if "carbon_intensity" in df.columns and pd.to_numeric(df["carbon_intensity"], errors="coerce").notna().any():
        return "carbon_intensity", "Grid carbon intensity (g CO2eq/kWh)"
    return None, None


def choose_resample_rule(dt: pd.Series, n_points: int) -> Optional[str]:
    """Choose a readable aggregation frequency for long time ranges."""
    dt = pd.to_datetime(dt, errors="coerce").dropna()
    if len(dt) < 2 or n_points <= 250:
        return None
    span = dt.max() - dt.min()
    if span <= pd.Timedelta(days=7):
        return None
    if span <= pd.Timedelta(days=120):
        return "D"
    if span <= pd.Timedelta(days=450):
        return "D"
    if span <= pd.Timedelta(days=3 * 365):
        return "W"
    return "MS"


def clean_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def format_date_axis(ax, dt: pd.Series):
    locator = mdates.AutoDateLocator(minticks=4, maxticks=9)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    ax.figure.autofmt_xdate(rotation=0, ha="center")


def plot_adaptive_lines(
    df: pd.DataFrame,
    y_cols: Sequence[str],
    *,
    title: str,
    ylabel: str,
    source: str,
    secondary_col: Optional[str] = None,
    secondary_ylabel: Optional[str] = None,
    threshold_lines: Sequence[float] = (),
    figsize: tuple[float, float] = (12, 5),
):
    """Plot raw, resampled, or representative-day data depending on the input."""
    if df is None or len(df) == 0 or not y_cols:
        print(f"Skipping plot: {title} — no plottable data.")
        return

    data = df.copy()
    for col in list(y_cols) + ([secondary_col] if secondary_col else []):
        if col is not None and col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")

    year_avg = is_year_average_like(data, source=source)
    if year_avg or not has_datetime(data):
        labels = representative_labels(data)
        x = np.arange(len(data))
        # Cap figure height to prevent matplotlib pixel limit errors (2^16 ≈ 65k).
        # Matplotlib's RendererAgg limits each dimension to <65536 pixels.
        # At typical DPI (80-100), 6 inches ≈ 600 pixels. Max 100 inches ≈ 10k pixels for safety.
        fig_height = min(10, 4 + 0.08 * len(data))  # ~0.08 inches per point, max 10 inches total
        fig, ax = plt.subplots(figsize=(figsize[0], fig_height))
        for col in y_cols:
            if col not in data.columns:
                continue
            ax.plot(x, data[col].to_numpy(dtype=float), marker="o", linewidth=1.8, label=col)

        for threshold in threshold_lines:
            ax.axhline(threshold, linestyle="--", linewidth=1.1, label=f"{threshold:g} kg CO2eq/kg H2 threshold")

        ax.set_title(title + " — representative periods" if year_avg else title)
        ax.set_xlabel("Representative period" if year_avg else "Timeslice")
        ax.set_ylabel(ylabel)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=35, ha="right")
        ax.grid(True, linestyle="--", alpha=0.35)
        ax.legend(framealpha=0.85)

        if secondary_col and secondary_col in data.columns:
            ax2 = ax.twinx()
            ax2.plot(x, data[secondary_col].to_numpy(dtype=float), linestyle=":", linewidth=1.3, label=secondary_ylabel or secondary_col)
            ax2.set_ylabel(secondary_ylabel or secondary_col)
            ax2.legend(loc="upper left", framealpha=0.75)
        
        try:
            plt.tight_layout()
        except ValueError as e:
            # If tight_layout fails due to size constraints, just use subplots_adjust instead
            if "too large" in str(e):
                fig.subplots_adjust(left=0.1, right=0.95, top=0.92, bottom=0.15, hspace=0.3)
            else:
                raise
        plt.show()
        return

    data = data.dropna(subset=["datetime"]).sort_values("datetime")
    dt = as_datetime_series(data)
    rule = choose_resample_rule(dt, len(data))

    plot_data = data.set_index("datetime")
    if rule is not None:
        numeric_cols = [c for c in list(y_cols) + ([secondary_col] if secondary_col else []) if c is not None and c in plot_data.columns]
        plot_data = plot_data[numeric_cols].resample(rule).mean().dropna(how="all")
        plot_note = f"{rule} mean from {len(data):,} slices"
        marker = None
    else:
        plot_note = f"raw slices (n={len(data):,})"
        marker = "o" if len(data) <= 120 else None

    fig, ax = plt.subplots(figsize=figsize)
    for col in y_cols:
        if col not in plot_data.columns:
            continue
        ax.plot(plot_data.index, plot_data[col], marker=marker, linewidth=1.6, markersize=3, label=col)

    for threshold in threshold_lines:
        ax.axhline(threshold, linestyle="--", linewidth=1.1, label=f"{threshold:g} kg CO2eq/kg H2 threshold")

    ax.set_title(f"{title} — {plot_note}")
    ax.set_xlabel("Time")
    ax.set_ylabel(ylabel)
    ax.grid(True, linestyle="--", alpha=0.35)
    format_date_axis(ax, pd.Series(plot_data.index))
    ax.legend(framealpha=0.85)

    if secondary_col and secondary_col in plot_data.columns:
        ax2 = ax.twinx()
        ax2.plot(plot_data.index, plot_data[secondary_col], linestyle=":", linewidth=1.2, label=secondary_ylabel or secondary_col)
        ax2.set_ylabel(secondary_ylabel or secondary_col)
        ax2.legend(loc="upper left", framealpha=0.75)

    plt.tight_layout()
    plt.show()


def duration_curve(series: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    s = pd.to_numeric(series, errors="coerce").dropna().sort_values().to_numpy()
    if len(s) == 0:
        return np.array([]), np.array([])
    pct = np.linspace(0, 100, len(s))
    return pct, s


## Plot 1 — custom-grid LCA GWP100

For short ranges this gives the raw timeslices. For long ranges it automatically aggregates to a readable mean line. For `year_average` it switches to representative-period labels instead of forcing a misleading dense time axis.


In [ ]:
if grid_df_results is None or grid_df_results.empty:
    print("Skipping custom-grid plot — no results loaded.")
else:
    tech_cols = grid_tech_columns(grid_df_results)
    secondary_col, secondary_ylabel = choose_secondary_grid_col(grid_df_results)
    print("Detected custom-grid technology columns:", tech_cols)

    plot_adaptive_lines(
        grid_df_results,
        tech_cols,
        title=f"Custom-grid hydrogen GWP100 ({cfg.METHOD_MODE} method)",
        ylabel="GWP100 (kg CO2eq / kg H2)",
        source="grid",
        secondary_col=secondary_col,
        secondary_ylabel=secondary_ylabel,
        threshold_lines=GWP_THRESHOLDS,
        figsize=(12, 5),
    )


## Plot 2 — wind/grid electrolyser GWP100

This handles `blended`, `switching`, and `both`. When both modes are loaded, each line label includes both the mode and the electrolyser technology.


In [ ]:
def plot_wind_gwp_adaptive(wind_df: Optional[pd.DataFrame]):
    if wind_df is None or wind_df.empty:
        print("Skipping wind/grid GWP plot — no results loaded.")
        return
    required = {"gwp100_kgco2e_per_kg_h2"}
    if not required.issubset(wind_df.columns):
        print("Skipping wind/grid GWP plot — required GWP column not found.")
        print("Available columns:", list(wind_df.columns))
        return

    data = wind_df.copy()
    if "electrolyser_tech" not in data.columns:
        data["electrolyser_tech"] = "electrolyser"
    if "wind_lca_mode" not in data.columns:
        data["wind_lca_mode"] = getattr(cfg, "WIND_LCA_MODE", "wind_grid")

    # Build a temporary wide table so the generic adaptive function can plot several lines.
    label_col = "series_label"
    data[label_col] = data["wind_lca_mode"].astype(str) + " — " + data["electrolyser_tech"].astype(str)

    if has_datetime(data):
        wide = data.pivot_table(
            index="datetime",
            columns=label_col,
            values="gwp100_kgco2e_per_kg_h2",
            aggfunc="mean",
        ).reset_index()
        # Preserve representative metadata if available by taking the first row per datetime.
        meta_cols = [c for c in data.columns if c not in ["gwp100_kgco2e_per_kg_h2", label_col] and c not in wide.columns]
        if meta_cols and is_year_average_like(data, source="wind"):
            meta = data.sort_values("datetime").groupby("datetime", as_index=False)[meta_cols].first()
            wide = wide.merge(meta, on="datetime", how="left")
    else:
        data["row_id"] = data.groupby(label_col).cumcount()
        wide = data.pivot_table(
            index="row_id",
            columns=label_col,
            values="gwp100_kgco2e_per_kg_h2",
            aggfunc="mean",
        ).reset_index()

    y_cols = [c for c in wide.columns if c not in {"datetime", "row_id"} and pd.api.types.is_numeric_dtype(wide[c])]
    plot_adaptive_lines(
        wide,
        y_cols,
        title=f"Wind/grid electrolyser GWP100 ({cfg.WIND_LCA_MODE} mode setting)",
        ylabel="GWP100 (kg CO2eq / kg H2)",
        source="wind",
        threshold_lines=GWP_THRESHOLDS,
        figsize=(12, 5),
    )


plot_wind_gwp_adaptive(wind_df_results)


## Plot 3 — wind/grid operating diagnostics

The diagnostic plot changes with the mode:

- For blended operation, it shows wind fraction versus GWP100.
- For switching operation over short ranges, it shows the wind/grid source sequence.
- For switching operation over long ranges, it summarises the share of wind operation by month.


In [ ]:
def plot_blended_diagnostic(df: pd.DataFrame):
    if "wind_fraction" not in df.columns or "gwp100_kgco2e_per_kg_h2" not in df.columns:
        return False
    data = df[df["wind_lca_mode"].astype(str).eq("blended")].copy() if "wind_lca_mode" in df.columns else df.copy()
    if data.empty:
        return False
    if "electrolyser_tech" not in data.columns:
        data["electrolyser_tech"] = "electrolyser"

    fig, ax = plt.subplots(figsize=(7, 5))
    for tech, tdf in data.groupby("electrolyser_tech"):
        plot_df = tdf[["wind_fraction", "gwp100_kgco2e_per_kg_h2"]].apply(pd.to_numeric, errors="coerce").dropna()
        if plot_df.empty:
            continue
        # Limit points only for rendering speed/readability; summary stats still use all rows elsewhere.
        if len(plot_df) > 3000:
            plot_df = plot_df.sample(3000, random_state=1).sort_values("wind_fraction")
        ax.scatter(plot_df["wind_fraction"], plot_df["gwp100_kgco2e_per_kg_h2"], alpha=0.55, s=14, label=tech)

    ax.set_title("Blended operation: wind fraction versus GWP100")
    ax.set_xlabel("Wind fraction of electrolyser electricity input")
    ax.set_ylabel("GWP100 (kg CO2eq / kg H2)")
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend(title="Electrolyser tech", framealpha=0.85)
    plt.tight_layout()
    plt.show()
    return True


def plot_switching_diagnostic(df: pd.DataFrame):
    if "electricity_source" not in df.columns:
        return False
    data = df[df["wind_lca_mode"].astype(str).eq("switching")].copy() if "wind_lca_mode" in df.columns else df.copy()
    if data.empty:
        return False
    if "electrolyser_tech" not in data.columns:
        data["electrolyser_tech"] = "electrolyser"

    # Use the first tech for a clean source-sequence diagnostic.
    first_tech = list(data["electrolyser_tech"].dropna().unique())[0]
    src = data[data["electrolyser_tech"] == first_tech].copy()

    if has_datetime(src) and not is_year_average_like(src, source="wind"):
        src["datetime"] = as_datetime_series(src)
        span = src["datetime"].max() - src["datetime"].min()
        if span > pd.Timedelta(days=45):
            monthly = (
                src.assign(is_wind=src["electricity_source"].astype(str).str.lower().eq("wind"))
                   .set_index("datetime")["is_wind"]
                   .resample("MS")
                   .mean()
                   .dropna()
            )
            fig, ax = plt.subplots(figsize=(12, 4))
            ax.plot(monthly.index, monthly.values, marker="o")
            ax.set_title(f"Switching operation: monthly share of wind operation — {first_tech}")
            ax.set_xlabel("Month")
            ax.set_ylabel("Share of slices using wind")
            ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
            ax.grid(True, linestyle="--", alpha=0.35)
            format_date_axis(ax, pd.Series(monthly.index))
            plt.tight_layout()
            plt.show()
            return True

        src = src.sort_values("datetime")
        x = src["datetime"]
        y = src["electricity_source"].astype(str).str.lower().map({"grid": 0, "wind": 1})
        fig, ax = plt.subplots(figsize=(12, 3.2))
        ax.step(x, y, where="post")
        ax.scatter(x, y, s=12)
        ax.set_title(f"Switching operation: electricity source sequence — {first_tech}")
        ax.set_xlabel("Time")
        ax.set_ylabel("Source")
        ax.set_yticks([0, 1])
        ax.set_yticklabels(["grid", "wind"])
        ax.grid(True, axis="x", linestyle="--", alpha=0.25)
        format_date_axis(ax, x)
        plt.tight_layout()
        plt.show()
        return True

    # Categorical fallback for year_average or no datetime.
    labels = representative_labels(src)
    y = src["electricity_source"].astype(str).str.lower().map({"grid": 0, "wind": 1})
    fig, ax = plt.subplots(figsize=(12, 3.2))
    ax.plot(np.arange(len(src)), y, marker="o", linewidth=1.2)
    ax.set_title(f"Switching operation: electricity source by representative period — {first_tech}")
    ax.set_xlabel("Representative period")
    ax.set_ylabel("Source")
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["grid", "wind"])
    ax.set_xticks(np.arange(len(src)))
    ax.set_xticklabels(labels, rotation=35, ha="right")
    ax.grid(True, axis="x", linestyle="--", alpha=0.25)
    plt.tight_layout()
    plt.show()
    return True


if wind_df_results is None or wind_df_results.empty:
    print("Skipping wind/grid diagnostics — no results loaded.")
else:
    shown_any = False
    shown_any = plot_blended_diagnostic(wind_df_results) or shown_any
    shown_any = plot_switching_diagnostic(wind_df_results) or shown_any
    if not shown_any:
        print("No wind/grid diagnostic columns found. Available columns:")
        print(list(wind_df_results.columns))


## Plot 4 — duration curves

Duration curves are useful for threshold questions such as: “what fraction of operation is below 2 kg CO₂eq/kg H₂?” They work for any time range, including a full year.


In [ ]:
def plot_duration_curves_from_grid(df: Optional[pd.DataFrame]):
    if df is None or df.empty:
        return False
    tech_cols = grid_tech_columns(df)
    if not tech_cols:
        return False
    fig, ax = plt.subplots(figsize=(8, 5))
    for col in tech_cols:
        pct, values = duration_curve(df[col])
        if len(values):
            ax.plot(pct, values, label=f"custom-grid — {col}")
    for threshold in GWP_THRESHOLDS:
        ax.axhline(threshold, linestyle="--", linewidth=1.1, label=f"{threshold:g} kg threshold")
    ax.set_title("Custom-grid GWP100 duration curve")
    ax.set_xlabel("Share of operating / representative periods (%)")
    ax.set_ylabel("GWP100 (kg CO2eq / kg H2)")
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend(framealpha=0.85)
    plt.tight_layout()
    plt.show()
    return True


def plot_duration_curves_from_wind(df: Optional[pd.DataFrame]):
    if df is None or df.empty or "gwp100_kgco2e_per_kg_h2" not in df.columns:
        return False
    data = df.copy()
    if "electrolyser_tech" not in data.columns:
        data["electrolyser_tech"] = "electrolyser"
    if "wind_lca_mode" not in data.columns:
        data["wind_lca_mode"] = getattr(cfg, "WIND_LCA_MODE", "wind_grid")

    fig, ax = plt.subplots(figsize=(8, 5))
    for (mode, tech), tdf in data.groupby(["wind_lca_mode", "electrolyser_tech"]):
        pct, values = duration_curve(tdf["gwp100_kgco2e_per_kg_h2"])
        if len(values):
            ax.plot(pct, values, label=f"{mode} — {tech}")
    for threshold in GWP_THRESHOLDS:
        ax.axhline(threshold, linestyle="--", linewidth=1.1, label=f"{threshold:g} kg threshold")
    ax.set_title("Wind/grid GWP100 duration curve")
    ax.set_xlabel("Share of operating / representative periods (%)")
    ax.set_ylabel("GWP100 (kg CO2eq / kg H2)")
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend(framealpha=0.85)
    plt.tight_layout()
    plt.show()
    return True

shown = False
shown = plot_duration_curves_from_grid(grid_df_results) or shown
shown = plot_duration_curves_from_wind(wind_df_results) or shown
if not shown:
    print("No data available for duration curves.")


## Optional full-range heatmap

This is only useful when you have real datetime slices across many days. It is skipped automatically for `single`, very short ranges, and `year_average` representative-day outputs.


In [ ]:
def plot_timeslice_heatmap(df: Optional[pd.DataFrame], value_col: str, *, title: str):
    if df is None or df.empty or value_col not in df.columns or not has_datetime(df):
        return False
    if is_year_average_like(df, source="wind") or is_year_average_like(df, source="grid"):
        print(f"Skipping heatmap for {title} — representative-period output is better shown categorically.")
        return False

    data = df[["datetime", value_col]].copy()
    data["datetime"] = as_datetime_series(data)
    data[value_col] = pd.to_numeric(data[value_col], errors="coerce")
    data = data.dropna().sort_values("datetime")
    if data.empty:
        return False
    span = data["datetime"].max() - data["datetime"].min()
    if span < pd.Timedelta(days=5):
        print(f"Skipping heatmap for {title} — range is short; line plot is clearer.")
        return False

    data["date"] = data["datetime"].dt.date
    data["slot"] = data["datetime"].dt.hour + data["datetime"].dt.minute / 60.0
    heatmap_data = data.pivot_table(index="date", columns="slot", values=value_col, aggfunc="mean")
    if heatmap_data.shape[0] < 5 or heatmap_data.shape[1] < 2:
        return False

    fig, ax = plt.subplots(figsize=(12, 7))
    im = ax.imshow(heatmap_data.to_numpy(), aspect="auto", origin="lower")
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("GWP100 (kg CO2eq / kg H2)")

    # x ticks: hours only, no overcrowding.
    slots = np.array(heatmap_data.columns, dtype=float)
    tick_idx = np.linspace(0, len(slots) - 1, min(9, len(slots))).astype(int)
    ax.set_xticks(tick_idx)
    ax.set_xticklabels([f"{slots[i]:g}" for i in tick_idx])

    # y ticks: about 8 evenly spaced dates.
    dates = list(heatmap_data.index)
    y_idx = np.linspace(0, len(dates) - 1, min(8, len(dates))).astype(int)
    ax.set_yticks(y_idx)
    ax.set_yticklabels([pd.to_datetime(dates[i]).strftime("%d %b") for i in y_idx])

    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Date")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return True

shown = False
if grid_df_results is not None and not grid_df_results.empty:
    tech_cols = grid_tech_columns(grid_df_results)
    if tech_cols:
        shown = plot_timeslice_heatmap(
            grid_df_results.reset_index(drop=True),
            tech_cols[0],
            title=f"Custom-grid GWP100 heatmap — {tech_cols[0]}",
        ) or shown

if wind_df_results is not None and not wind_df_results.empty and "gwp100_kgco2e_per_kg_h2" in wind_df_results.columns:
    tmp = wind_df_results.copy()
    if "electrolyser_tech" in tmp.columns:
        tmp = tmp[tmp["electrolyser_tech"] == tmp["electrolyser_tech"].dropna().iloc[0]]
    if "wind_lca_mode" in tmp.columns:
        tmp = tmp[tmp["wind_lca_mode"] == tmp["wind_lca_mode"].dropna().iloc[0]]
    shown = plot_timeslice_heatmap(
        tmp,
        "gwp100_kgco2e_per_kg_h2",
        title="Wind/grid GWP100 heatmap",
    ) or shown

if not shown:
    print("No suitable long datetime range found for a heatmap.")


## Combined summary table

The summary includes mean/min/max and threshold shares. This is usually the cleanest place to report “below 2 kg CO₂eq/kg H₂ for X% of periods.”


In [ ]:
def threshold_share(series: pd.Series, threshold: float) -> float:
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return np.nan
    return 100.0 * float((s <= threshold).mean())

rows = []

if grid_df_results is not None and not grid_df_results.empty:
    for tech in grid_tech_columns(grid_df_results):
        s = pd.to_numeric(grid_df_results[tech], errors="coerce").dropna()
        if s.empty:
            continue
        row = {
            "source": "custom_grid",
            "mode": cfg.METHOD_MODE,
            "time_mode": cfg.GRID_TIME_MODE,
            "technology": tech,
            "n_periods": int(s.notna().sum()),
            "mean_gwp100": s.mean(),
            "median_gwp100": s.median(),
            "p10_gwp100": s.quantile(0.10),
            "p90_gwp100": s.quantile(0.90),
            "min_gwp100": s.min(),
            "max_gwp100": s.max(),
        }
        for threshold in GWP_THRESHOLDS:
            row[f"share_below_{threshold:g}kg_%"] = threshold_share(s, threshold)
        rows.append(row)

if wind_df_results is not None and not wind_df_results.empty and "gwp100_kgco2e_per_kg_h2" in wind_df_results.columns:
    data = wind_df_results.copy()
    if "electrolyser_tech" not in data.columns:
        data["electrolyser_tech"] = "electrolyser"
    if "wind_lca_mode" not in data.columns:
        data["wind_lca_mode"] = getattr(cfg, "WIND_LCA_MODE", "wind_grid")

    for (mode, tech), tdf in data.groupby(["wind_lca_mode", "electrolyser_tech"]):
        s = pd.to_numeric(tdf["gwp100_kgco2e_per_kg_h2"], errors="coerce").dropna()
        if s.empty:
            continue
        row = {
            "source": "wind_grid",
            "mode": mode,
            "time_mode": cfg.GRID_TIME_MODE,
            "technology": tech,
            "n_periods": int(s.notna().sum()),
            "mean_gwp100": s.mean(),
            "median_gwp100": s.median(),
            "p10_gwp100": s.quantile(0.10),
            "p90_gwp100": s.quantile(0.90),
            "min_gwp100": s.min(),
            "max_gwp100": s.max(),
        }
        for threshold in GWP_THRESHOLDS:
            row[f"share_below_{threshold:g}kg_%"] = threshold_share(s, threshold)
        rows.append(row)

if rows:
    summary = pd.DataFrame(rows).set_index(["source", "mode", "time_mode", "technology"]).sort_index()
    display(summary.round(3))
else:
    print("No results to summarise. Run the building-block notebooks first.")
